# AI651 Assignment 1 — Task 2
## End-to-end Autoformer leaderboard notebook

Run this notebook **top-to-bottom on Kaggle with a T4 GPU**.

It performs data sanity checks, forecasting-oriented EDA, leakage-safe chronological validation, a genuine Autoformer-style model with progressive series decomposition + FFT time-delay aggregation, compact hyperparameter screening, the required external-data ablation, final **2-seed × 3-fold** evaluation, full-history retraining, and generation of the final 168 predictions.

The notebook intentionally uses staged evidence-driven screening rather than a huge brute-force grid.

In [ ]:
import os, glob, json, math, random, time, copy, warnings
from dataclasses import dataclass, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import periodogram
from scipy.stats import skew, kurtosis
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available(): torch.set_float32_matmul_precision('high')
print('PyTorch:',torch.__version__,'| Device:',DEVICE)
if torch.cuda.is_available(): print('GPU:',torch.cuda.get_device_name(0))

RUN_SMOKE_TEST=True
RUN_FULL_PIPELINE=True
PRED_LEN=168
SCREEN_ORIGINS=[43320,43488]
FINAL_ORIGINS=[43152,43320,43488]
SCREEN_SEEDS=[2026]
FINAL_SEEDS=[2026,3407]
SCREEN_MAX_EPOCHS=4
SCREEN_PATIENCE=1
SCREEN_TRAIN_STRIDE=8
FINAL_MAX_EPOCHS=10
FINAL_PATIENCE=2
FINAL_TRAIN_STRIDE=4
FINAL_RETRAIN_STRIDE=2
NUM_WORKERS=2 if torch.cuda.is_available() else 0
USE_AMP=bool(torch.cuda.is_available())
OUTPUT_DIR=Path('/kaggle/working/task2_autoformer') if Path('/kaggle/working').exists() else Path('./task2_autoformer')
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
print('Output:',OUTPUT_DIR)

## 1. Load and validate the three CSV files
Attach the dataset containing `student_train.csv`, `student_test.csv`, and `optional_external_data.csv`.

In [ ]:
def find_file(filename):
    hits=[]
    for root in ['/kaggle/input','/kaggle/working','/mnt/data','.']:
        if os.path.exists(root): hits += glob.glob(os.path.join(root,'**',filename),recursive=True)
    hits=list(dict.fromkeys(hits))
    if not hits: raise FileNotFoundError(filename)
    hits.sort(key=lambda p:(0 if '/kaggle/input/' in p else 1 if '/mnt/data/' in p else 2,len(p)))
    print(filename,':',hits[0]); return hits[0]

train_df=pd.read_csv(find_file('student_train.csv'))
test_df=pd.read_csv(find_file('student_test.csv'))
ext_df=pd.read_csv(find_file('optional_external_data.csv'))
print('train',train_df.shape,'test',test_df.shape,'external',ext_df.shape)

assert list(train_df.columns)==['time_idx','value']
assert list(test_df.columns)==['time_idx','value']
assert len(train_df)==43656 and len(test_df)==168 and len(ext_df)==43824
assert train_df.time_idx.tolist()==list(range(1,43657))
assert test_df.time_idx.tolist()==list(range(43657,43825))
assert ext_df.time_idx.tolist()==list(range(1,43825))
assert train_df.value.notna().all() and test_df.value.isna().all() and (train_df.value>=0).all()
EXOG_COLS=[c for c in ext_df.columns if c!='time_idx']
CONT_COLS=['feature_A','feature_B','feature_C','feature_D','feature_E','feature_F']
BINARY_COLS=['feature_G','feature_H','feature_I','feature_J']
assert len(EXOG_COLS)==10
assert set(np.unique(ext_df[BINARY_COLS].values)).issubset({0,1})
assert np.all(ext_df[BINARY_COLS].sum(axis=1).values==1)
assert ext_df[EXOG_COLS].notna().all().all()
print('All structural checks passed.')

## 2. Forecasting-oriented EDA
These diagnostics inspect heavy tails, level-dependent variance, autocorrelation, local recurrence, spectral structure, decomposition scales, and external variables.

In [ ]:
y=train_df.value.to_numpy(np.float64)
summary={'n':len(y),'min':float(y.min()),'max':float(y.max()),'mean':float(y.mean()),'median':float(np.median(y)),'std':float(y.std()),'skew':float(skew(y)),'excess_kurtosis':float(kurtosis(y))}
for q in [.90,.95,.99,.999]: summary[f'q{q}']=float(np.quantile(y,q))
display(pd.DataFrame([summary]).T.rename(columns={0:'value'}))

rm=pd.Series(y).rolling(168).mean(); rs=pd.Series(y).rolling(168).std()
ylog=np.log1p(y); lrm=pd.Series(ylog).rolling(168).mean(); lrs=pd.Series(ylog).rolling(168).std()
print('raw rolling mean/std corr:',pd.concat([rm,rs],axis=1).dropna().corr().iloc[0,1])
print('log rolling mean/std corr:',pd.concat([lrm,lrs],axis=1).dropna().corr().iloc[0,1])

plt.figure(figsize=(14,4)); plt.plot(train_df.time_idx,y,lw=.7); plt.title('Full target series'); plt.tight_layout(); plt.show()
plt.figure(figsize=(14,4)); plt.plot(train_df.time_idx.iloc[-1008:],y[-1008:],lw=1); plt.title('Last six 168-step horizons'); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4)); plt.hist(y,bins=80); plt.title('Target distribution — raw'); plt.tight_layout(); plt.show()
plt.figure(figsize=(8,4)); plt.hist(ylog,bins=80); plt.title('Target distribution — log1p'); plt.tight_layout(); plt.show()

In [ ]:
def acf_numpy(x,max_lag):
    x=np.asarray(x,np.float64); x=x-x.mean(); den=np.dot(x,x); out=np.empty(max_lag+1); out[0]=1
    for lag in range(1,max_lag+1): out[lag]=np.dot(x[:-lag],x[lag:])/den
    return out

gacf=acf_numpy(y,336)
lags=[1,2,5,12,24,48,72,96,168,336]
display(pd.DataFrame({'lag':lags,'acf':[gacf[l] for l in lags]}))
plt.figure(figsize=(12,4)); plt.plot(np.arange(337),gacf); plt.axhline(0,lw=.7); plt.title('Global ACF'); plt.tight_layout(); plt.show()

rows=[]
for start in range(0,len(y)-672+1,672): rows.append(acf_numpy(y[start:start+672],168)[1:])
local_acf=np.vstack(rows)
plt.figure(figsize=(12,6)); plt.imshow(local_acf,aspect='auto',interpolation='nearest',extent=[1,168,len(rows),0]); plt.colorbar(label='local ACF'); plt.title('Local ACF — 672-point windows'); plt.xlabel('lag'); plt.tight_layout(); plt.show()
print('Local lag-24 ACF range:',local_acf[:,23].min(),local_acf[:,23].max())

freq,power=periodogram(y-y.mean()); m=freq>0; freq,power=freq[m],power[m]; top=np.argsort(power)[-15:][::-1]
display(pd.DataFrame({'frequency':freq[top],'approx_period':1/freq[top],'power':power[top]}))

In [ ]:
def centered_ma(x,k):
    p=(k-1)//2; xp=np.pad(x,(p,p),mode='edge'); return np.convolve(xp,np.ones(k)/k,mode='valid')
for k in [13,25,49,97]:
    seg=y[-672:]; tr=centered_ma(seg,k)
    plt.figure(figsize=(14,4)); plt.plot(seg,label='target',lw=1); plt.plot(tr,label=f'MA trend k={k}',lw=1); plt.legend(); plt.tight_layout(); plt.show()

joined=ext_df.iloc[:len(train_df)].copy(); joined['target']=y
pearson=joined[EXOG_COLS+['target']].corr()['target'].drop('target').sort_values()
display(pearson.to_frame('contemporaneous_pearson'))

def best_lead_corr(feature,target,max_lead=168):
    best=None
    for lead in range(max_lead+1):
        a,b=(feature,target) if lead==0 else (feature[:-lead],target[lead:])
        c=np.corrcoef(a,b)[0,1]
        cand=(abs(c),lead,c)
        if np.isfinite(c) and (best is None or cand[0]>best[0]): best=cand
    return best[1],best[2]

display(pd.DataFrame([{'feature':c,'best_lead':best_lead_corr(joined[c].to_numpy(float),y)[0],'corr':best_lead_corr(joined[c].to_numpy(float),y)[1]} for c in CONT_COLS]))
display(pd.DataFrame([{'state':c,'count':int((joined[c]==1).sum()),'mean_target':float(y[joined[c].to_numpy()==1].mean()),'median_target':float(np.median(y[joined[c].to_numpy()==1]))} for c in BINARY_COLS]))

hist=ext_df.iloc[:len(train_df)]; fut=ext_df.iloc[len(train_df):]
display(pd.DataFrame({'history_fraction':hist[BINARY_COLS].mean(),'future168_fraction':fut[BINARY_COLS].mean()}))

## 3. Genuine Autoformer-style implementation
The target is progressively decomposed, and sequence mixing uses FFT correlation followed by weighted top-delay aggregation rather than vanilla point-wise attention. Future A–J covariates are supplied to the decoder, while future targets are never supplied.

In [ ]:
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

class MovingAverage(nn.Module):
    def __init__(self,k):
        super().__init__(); assert k%2==1; self.k=k; self.pool=nn.AvgPool1d(k,stride=1)
    def forward(self,x):
        p=(self.k-1)//2; z=torch.cat([x[:,:1,:].expand(-1,p,-1),x,x[:,-1:,:].expand(-1,p,-1)],dim=1)
        return self.pool(z.transpose(1,2)).transpose(1,2)

class SeriesDecomp(nn.Module):
    def __init__(self,k): super().__init__(); self.ma=MovingAverage(k)
    def forward(self,x):
        t=self.ma(x); return x-t,t

class SeasonalLayerNorm(nn.Module):
    def __init__(self,d): super().__init__(); self.norm=nn.LayerNorm(d)
    def forward(self,x):
        z=self.norm(x); return z-z.mean(dim=1,keepdim=True)

class AutoCorrelation(nn.Module):
    def __init__(self,factor=3,dropout=.1): super().__init__(); self.factor=factor; self.drop=nn.Dropout(dropout)
    def forward(self,q,k,v):
        B,L,H,D=q.shape; S=k.shape[1]
        if L>S:
            pad=L-S
            k=torch.cat([k,torch.zeros(B,pad,H,D,device=k.device,dtype=k.dtype)],1)
            v=torch.cat([v,torch.zeros(B,pad,H,D,device=v.device,dtype=v.dtype)],1)
        else: k=k[:,:L]; v=v[:,:L]
        qf=torch.fft.rfft(q.permute(0,2,3,1).contiguous().float(),dim=-1)
        kf=torch.fft.rfft(k.permute(0,2,3,1).contiguous().float(),dim=-1)
        corr=torch.fft.irfft(qf*torch.conj(kf),n=L,dim=-1)
        vals=v.permute(0,2,3,1).contiguous(); topk=max(1,min(L,int(self.factor*math.log(max(L,2)))))
        score=corr.mean(dim=(1,2)); weights,delays=torch.topk(score,topk,dim=-1); weights=torch.softmax(weights,dim=-1).to(vals.dtype)
        doubled=torch.cat([vals,vals],dim=-1); base=torch.arange(L,device=vals.device)[None,None,None,:].expand(B,H,D,L); out=torch.zeros_like(vals)
        for j in range(topk):
            idx=base+delays[:,j].view(B,1,1,1); out += torch.gather(doubled,-1,idx)*weights[:,j].view(B,1,1,1)
        return self.drop(out).permute(0,3,1,2).contiguous()

class AutoCorrelationLayer(nn.Module):
    def __init__(self,d_model,n_heads,factor,dropout):
        super().__init__(); assert d_model%n_heads==0; self.h=n_heads; self.d=d_model//n_heads
        self.q=nn.Linear(d_model,d_model); self.k=nn.Linear(d_model,d_model); self.v=nn.Linear(d_model,d_model); self.o=nn.Linear(d_model,d_model); self.ac=AutoCorrelation(factor,dropout)
    def forward(self,Q,K,V):
        B,L,_=Q.shape; S=K.shape[1]; H=self.h
        q=self.q(Q).view(B,L,H,self.d); k=self.k(K).view(B,S,H,self.d); v=self.v(V).view(B,S,H,self.d)
        return self.o(self.ac(q,k,v).reshape(B,L,-1))

class EncoderLayer(nn.Module):
    def __init__(self,d,h,dff,k,factor,drop):
        super().__init__(); self.mix=AutoCorrelationLayer(d,h,factor,drop); self.d1=SeriesDecomp(k); self.d2=SeriesDecomp(k); self.f1=nn.Conv1d(d,dff,1,bias=False); self.f2=nn.Conv1d(dff,d,1,bias=False); self.drop=nn.Dropout(drop)
    def forward(self,x):
        x,_=self.d1(x+self.drop(self.mix(x,x,x))); y=self.f2(self.drop(F.gelu(self.f1(x.transpose(1,2))))).transpose(1,2); x,_=self.d2(x+self.drop(y)); return x

class DecoderLayer(nn.Module):
    def __init__(self,d,h,dff,k,factor,drop):
        super().__init__(); self.selfmix=AutoCorrelationLayer(d,h,factor,drop); self.cross=AutoCorrelationLayer(d,h,factor,drop); self.d1=SeriesDecomp(k); self.d2=SeriesDecomp(k); self.d3=SeriesDecomp(k); self.f1=nn.Conv1d(d,dff,1,bias=False); self.f2=nn.Conv1d(dff,d,1,bias=False); self.tp=nn.Conv1d(d,1,3,padding=1,padding_mode='circular',bias=False); self.drop=nn.Dropout(drop)
    def forward(self,x,cross,trend):
        x,t1=self.d1(x+self.drop(self.selfmix(x,x,x))); x,t2=self.d2(x+self.drop(self.cross(x,cross,cross))); y=self.f2(self.drop(F.gelu(self.f1(x.transpose(1,2))))).transpose(1,2); x,t3=self.d3(x+self.drop(y)); return x,trend+self.tp((t1+t2+t3).transpose(1,2)).transpose(1,2)

class Embed(nn.Module):
    def __init__(self,d,exog_dim,drop):
        super().__init__(); self.t=nn.Conv1d(1,d,3,padding=1,padding_mode='circular',bias=False); self.x=nn.Linear(exog_dim,d,bias=False) if exog_dim else None; self.drop=nn.Dropout(drop)
    def forward(self,target,exog=None):
        z=self.t(target.transpose(1,2)).transpose(1,2)
        if self.x is not None: z=z+self.x(exog)
        return self.drop(z)

@dataclass
class Config:
    seq_len:int=336; label_len:int=84; pred_len:int=168; d_model:int=64; n_heads:int=4; e_layers:int=2; d_layers:int=1; d_ff:int=128; moving_avg:int=25; factor:int=3; dropout:float=.1; exog_dim:int=10

class Autoformer(nn.Module):
    def __init__(self,c):
        super().__init__(); self.c=c; self.initdec=SeriesDecomp(c.moving_avg); self.ee=Embed(c.d_model,c.exog_dim,c.dropout); self.de=Embed(c.d_model,c.exog_dim,c.dropout); self.enc=nn.ModuleList([EncoderLayer(c.d_model,c.n_heads,c.d_ff,c.moving_avg,c.factor,c.dropout) for _ in range(c.e_layers)]); self.enorm=SeasonalLayerNorm(c.d_model); self.dec=nn.ModuleList([DecoderLayer(c.d_model,c.n_heads,c.d_ff,c.moving_avg,c.factor,c.dropout) for _ in range(c.d_layers)]); self.dnorm=SeasonalLayerNorm(c.d_model); self.proj=nn.Linear(c.d_model,1)
    def forward(self,x_target,x_exog=None,decoder_exog=None):
        B=x_target.shape[0]; c=self.c; seas,trend=self.initdec(x_target); z=torch.zeros(B,c.pred_len,1,device=x_target.device,dtype=x_target.dtype); mean=x_target.mean(1,keepdim=True).expand(-1,c.pred_len,-1); decs=torch.cat([seas[:,-c.label_len:],z],1); dect=torch.cat([trend[:,-c.label_len:],mean],1)
        e=self.ee(x_target,x_exog)
        for layer in self.enc: e=layer(e)
        e=self.enorm(e); d=self.de(decs,decoder_exog); t=dect
        for layer in self.dec: d,t=layer(d,e,t)
        return (self.proj(self.dnorm(d))+t)[:,-c.pred_len:]

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 4. Leakage-safe windows and scaling
Target/external scalers for each fold are fit only on history before the forecast origin. Continuous A–F are standardized; binary G–J remain 0/1.

In [ ]:
Y_ALL=train_df.value.to_numpy(np.float32); X_ALL=ext_df[EXOG_COLS].to_numpy(np.float32)
class WindowDataset(Dataset):
    def __init__(self,y,x,cutoff,seq_len,pred_len,stride=1,max_windows=None):
        self.y=y; self.x=x; self.seq=seq_len; self.pred=pred_len; starts=np.arange(0,cutoff-seq_len-pred_len+1,stride,dtype=np.int64)
        if max_windows and len(starts)>max_windows: starts=starts[np.linspace(0,len(starts)-1,max_windows).round().astype(int)]
        self.starts=starts
    def __len__(self): return len(self.starts)
    def __getitem__(self,i):
        s=int(self.starts[i]); a=s+self.seq; b=a+self.pred
        return torch.from_numpy(self.y[s:a,None].copy()),torch.from_numpy(self.y[a:b,None].copy()),torch.from_numpy(self.x[s:a].copy()),torch.from_numpy(self.x[a:b].copy())

class ExogScaler:
    def __init__(self): self.cont=[EXOG_COLS.index(c) for c in CONT_COLS]
    def fit(self,x):
        self.mean=np.zeros(x.shape[1],np.float32); self.std=np.ones(x.shape[1],np.float32); self.mean[self.cont]=x[:,self.cont].mean(0); self.std[self.cont]=x[:,self.cont].std(0)+1e-6; return self
    def state_dict(self): return {'mean':self.mean.tolist(),'std':self.std.tolist(),'feature_names':EXOG_COLS}

def tstats(y,mode):
    z=y.astype(np.float64) if mode=='global_z' else np.log1p(y.astype(np.float64)); return float(z.mean()),float(z.std()+1e-6)
def tform(x,mode,mean,std):
    z=x if mode=='global_z' else torch.log1p(torch.clamp_min(x,0)); return (z-mean)/std
def tinv(z,mode,mean,std):
    x=z*std+mean; x=torch.expm1(x) if mode=='log1p_global_z' else x; return torch.clamp_min(x,0)
def fidx(mode): return [] if mode=='none' else list(range(len(EXOG_COLS)))
def sx(x,scaler,idx):
    if not idx:return None
    m=torch.tensor(scaler.mean[idx],dtype=x.dtype,device=x.device); s=torch.tensor(scaler.std[idx],dtype=x.dtype,device=x.device); return (x[:,:,idx]-m)/s

In [ ]:
def model_cfg(exp): return Config(seq_len=exp['seq_len'],label_len=exp['label_len'],d_model=exp['d_model'],n_heads=exp['n_heads'],e_layers=exp['e_layers'],d_layers=exp['d_layers'],d_ff=exp['d_ff'],moving_avg=exp['moving_avg'],factor=exp['factor'],dropout=exp['dropout'],exog_dim=len(fidx(exp['exog_mode'])))

def metrics(true,pred):
    true=np.asarray(true,float); pred=np.asarray(pred,float); return {'rmse':float(np.sqrt(np.mean((true-pred)**2))),'mae':float(np.mean(np.abs(true-pred))),'smape':float(np.mean(2*np.abs(true-pred)/(np.abs(true)+np.abs(pred)+1e-8))*100)}

def prep_batch(batch,exp,ym,ys,xs):
    hy,fy,hx,fx=[b.to(DEVICE,non_blocking=True) for b in batch]; hy=tform(hy,exp['target_mode'],ym,ys); fy=tform(fy,exp['target_mode'],ym,ys); idx=fidx(exp['exog_mode'])
    if idx:
        hx=sx(hx,xs,idx); fx=sx(fx,xs,idx); dx=torch.cat([hx[:,-exp['label_len']:],fx],1)
    else: hx=None; dx=None
    return hy,fy,hx,dx

@torch.no_grad()
def fold_pred(model,origin,exp,ym,ys,xs):
    seq=exp['seq_len']; hy=torch.tensor(Y_ALL[origin-seq:origin,None],dtype=torch.float32,device=DEVICE)[None]; hy=tform(hy,exp['target_mode'],ym,ys); idx=fidx(exp['exog_mode'])
    if idx:
        hx=torch.tensor(X_ALL[origin-seq:origin],dtype=torch.float32,device=DEVICE)[None]; fx=torch.tensor(X_ALL[origin:origin+168],dtype=torch.float32,device=DEVICE)[None]; hx=sx(hx,xs,idx); fx=sx(fx,xs,idx); dx=torch.cat([hx[:,-exp['label_len']:],fx],1)
    else:hx=None;dx=None
    with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP and DEVICE.type=='cuda'): p=model(hy,hx,dx)
    return np.clip(tinv(p.float(),exp['target_mode'],ym,ys).squeeze().cpu().numpy(),0,None),Y_ALL[origin:origin+168].astype(float)

def train_fold(exp,origin,seed,stride,max_epochs,patience,max_windows=None):
    set_seed(seed); ym,ys=tstats(Y_ALL[:origin],exp['target_mode']); xs=ExogScaler().fit(X_ALL[:origin]); ds=WindowDataset(Y_ALL,X_ALL,origin,exp['seq_len'],168,stride,max_windows); dl=DataLoader(ds,batch_size=exp['batch_size'],shuffle=True,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda'); model=Autoformer(model_cfg(exp)).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=exp['lr'],weight_decay=exp['weight_decay']); scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and DEVICE.type=='cuda'); best=float('inf'); state=None; bestep=0; bad=0; start=time.time()
    for ep in range(1,max_epochs+1):
        model.train(); losses=[]
        for batch in dl:
            hy,fy,hx,dx=prep_batch(batch,exp,ym,ys,xs); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP and DEVICE.type=='cuda'): pr=model(hy,hx,dx); loss=F.mse_loss(pr,fy)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); losses.append(loss.item())
        p,t=fold_pred(model,origin,exp,ym,ys,xs); r=metrics(t,p); print(f'ep {ep}: RMSE {r["rmse"]:.3f}')
        if r['rmse']<best: best=r['rmse']; bestep=ep; state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; bad=0
        else: bad+=1
        if bad>=patience: break
    model.load_state_dict(state); model.to(DEVICE); p,t=fold_pred(model,origin,exp,ym,ys,xs); r=metrics(t,p); out={'origin':origin,'seed':seed,'best_epoch':bestep,'n_params':count_params(model),'train_windows':len(ds),'elapsed_sec':time.time()-start,**r}; hse=(p-t)**2; del model
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return out,hse

def evaluate(name,exp,origins,seeds,stride,max_epochs,patience,max_windows=None):
    rows=[]
    for seed in seeds:
        for origin in origins:
            print(f'\n{name} | origin={origin} seed={seed}'); r,h=train_fold(exp,origin,seed,stride,max_epochs,patience,max_windows); rows.append({'experiment':name,'config_json':json.dumps(exp,sort_keys=True),**r}); HORIZON_RECORDS.append({'experiment':name,'origin':origin,'seed':seed,'sq_error':h}); print(r)
    return pd.DataFrame(rows)

def summarize(df): return df.groupby('experiment').agg(mean_rmse=('rmse','mean'),std_rmse=('rmse','std'),mean_mae=('mae','mean'),mean_smape=('smape','mean'),params=('n_params','first'),mean_best_epoch=('best_epoch','mean'),mean_elapsed_min=('elapsed_sec',lambda s:s.mean()/60)).sort_values('mean_rmse')
def choose(summary,tie=.005):
    s=summary.sort_values('mean_rmse'); best=s.iloc[0].mean_rmse; c=s[s.mean_rmse<=best*(1+tie)].sort_values(['params','mean_best_epoch','mean_rmse']); return c.index[0]
HORIZON_RECORDS=[]

## 5. Smoke test
This is only an end-to-end correctness test. It is not used for final model selection.

In [ ]:
BASE={'seq_len':336,'label_len':84,'d_model':64,'n_heads':4,'e_layers':2,'d_layers':1,'d_ff':128,'moving_avg':25,'factor':3,'dropout':.1,'target_mode':'global_z','exog_mode':'all','lr':1e-4,'weight_decay':1e-4,'batch_size':32}
if RUN_SMOKE_TEST:
    smoke={**BASE,'seq_len':168,'label_len':48,'d_model':16,'e_layers':1,'d_ff':32,'moving_avg':13,'factor':2,'batch_size':16}
    smoke_df=evaluate('SMOKE_ONLY',smoke,[43488],[2026],32,1,1,256); display(smoke_df); assert np.isfinite(smoke_df.rmse).all(); print('SMOKE TEST PASSED')

## 6. Staged search
Stage A selects raw vs log1p. Stage B selects context/decomposition. Stage C selects capacity. Then the exact selected architecture is compared with vs without external variables.

In [ ]:
all_screen=[]; cfgmap={}
if RUN_FULL_PIPELINE:
    cand={'T_raw':{**BASE,'target_mode':'global_z'},'T_log1p':{**BASE,'target_mode':'log1p_global_z'}}; frames=[]
    for n,e in cand.items(): cfgmap[n]=e; d=evaluate(n,e,SCREEN_ORIGINS,SCREEN_SEEDS,SCREEN_TRAIN_STRIDE,SCREEN_MAX_EPOCHS,SCREEN_PATIENCE); frames.append(d); all_screen.append(d)
    ts=summarize(pd.concat(frames)); display(ts); bn=choose(ts); best_mode=cfgmap[bn]['target_mode']; print('Selected transform:',bn)

    cand={'S_168_k25':{**BASE,'target_mode':best_mode,'seq_len':168,'label_len':84,'moving_avg':25},'S_336_k25':{**BASE,'target_mode':best_mode,'seq_len':336,'label_len':84,'moving_avg':25},'S_336_k49':{**BASE,'target_mode':best_mode,'seq_len':336,'label_len':84,'moving_avg':49},'S_672_k49':{**BASE,'target_mode':best_mode,'seq_len':672,'label_len':84,'moving_avg':49}}; frames=[]
    for n,e in cand.items(): cfgmap[n]=e; d=evaluate(n,e,SCREEN_ORIGINS,SCREEN_SEEDS,SCREEN_TRAIN_STRIDE,SCREEN_MAX_EPOCHS,SCREEN_PATIENCE); frames.append(d); all_screen.append(d)
    ss=summarize(pd.concat(frames)); display(ss); bn=choose(ss); structure=copy.deepcopy(cfgmap[bn]); print('Selected structure:',bn)

    cand={'C_compact32':{**structure,'d_model':32,'d_ff':64},'C_base64':{**structure,'d_model':64,'d_ff':128},'C_medium96':{**structure,'d_model':96,'d_ff':192}}; frames=[]
    for n,e in cand.items(): cfgmap[n]=e; d=evaluate(n,e,SCREEN_ORIGINS,SCREEN_SEEDS,SCREEN_TRAIN_STRIDE,SCREEN_MAX_EPOCHS,SCREEN_PATIENCE); frames.append(d); all_screen.append(d)
    cs=summarize(pd.concat(frames)); display(cs); bn=choose(cs); chosen=copy.deepcopy(cfgmap[bn]); print('Selected capacity:',bn,chosen)
    screening_df=pd.concat(all_screen,ignore_index=True); screening_df.to_csv(OUTPUT_DIR/'screening_results.csv',index=False); summarize(screening_df).to_csv(OUTPUT_DIR/'screening_summary.csv')

## 7. Required external-data ablation + final 2-seed × 3-fold evaluation

In [ ]:
if RUN_FULL_PIPELINE:
    variants={'FINAL_with_external':{**chosen,'exog_mode':'all'},'FINAL_without_external':{**chosen,'exog_mode':'none'}}; frames=[]
    for n,e in variants.items(): frames.append(evaluate(n,e,FINAL_ORIGINS,FINAL_SEEDS,FINAL_TRAIN_STRIDE,FINAL_MAX_EPOCHS,FINAL_PATIENCE))
    final_cv=pd.concat(frames,ignore_index=True); final_summary=summarize(final_cv); display(final_cv); display(final_summary); final_cv.to_csv(OUTPUT_DIR/'final_cv_results.csv',index=False); final_summary.to_csv(OUTPUT_DIR/'final_cv_summary.csv'); best_name=choose(final_summary); best_exp=copy.deepcopy(variants[best_name]); print('SELECTED FINAL:',best_name,best_exp)

    hrs=[]
    for n in variants:
        m=[r for r in HORIZON_RECORDS if r['experiment']==n and r['origin'] in FINAL_ORIGINS and r['seed'] in FINAL_SEEDS]; rm=np.sqrt(np.vstack([r['sq_error'] for r in m]).mean(0)); hrs += [{'experiment':n,'horizon_step':i+1,'rmse':float(v)} for i,v in enumerate(rm)]
        plt.figure(figsize=(12,4)); plt.plot(np.arange(1,169),rm); plt.title('Horizon-wise RMSE — '+n); plt.tight_layout(); plt.show()
    pd.DataFrame(hrs).to_csv(OUTPUT_DIR/'final_horizon_rmse.csv',index=False)

## 8. Retrain selected model on all 43,656 observed targets and predict test

In [ ]:
def train_full(exp,seed,epochs,stride):
    set_seed(seed); cutoff=len(Y_ALL); ym,ys=tstats(Y_ALL,exp['target_mode']); xs=ExogScaler().fit(X_ALL[:cutoff]); ds=WindowDataset(Y_ALL,X_ALL,cutoff,exp['seq_len'],168,stride); dl=DataLoader(ds,batch_size=exp['batch_size'],shuffle=True,num_workers=NUM_WORKERS,pin_memory=DEVICE.type=='cuda'); model=Autoformer(model_cfg(exp)).to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=exp['lr'],weight_decay=exp['weight_decay']); scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP and DEVICE.type=='cuda'); start=time.time()
    for ep in range(1,epochs+1):
        model.train(); losses=[]
        for batch in tqdm(dl,leave=False,desc=f'FINAL {ep}/{epochs}'):
            hy,fy,hx,dx=prep_batch(batch,exp,ym,ys,xs); opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP and DEVICE.type=='cuda'): pr=model(hy,hx,dx); loss=F.mse_loss(pr,fy)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update(); losses.append(loss.item())
        print('epoch',ep,'loss',np.mean(losses))
    return model,ym,ys,xs,time.time()-start

@torch.no_grad()
def predict_test(model,exp,ym,ys,xs):
    origin=len(Y_ALL); seq=exp['seq_len']; hy=torch.tensor(Y_ALL[origin-seq:origin,None],dtype=torch.float32,device=DEVICE)[None]; hy=tform(hy,exp['target_mode'],ym,ys); idx=fidx(exp['exog_mode'])
    if idx:
        hx=torch.tensor(X_ALL[origin-seq:origin],dtype=torch.float32,device=DEVICE)[None]; fx=torch.tensor(X_ALL[origin:origin+168],dtype=torch.float32,device=DEVICE)[None]; hx=sx(hx,xs,idx); fx=sx(fx,xs,idx); dx=torch.cat([hx[:,-exp['label_len']:],fx],1)
    else:hx=None;dx=None
    with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=USE_AMP and DEVICE.type=='cuda'): p=model(hy,hx,dx)
    return np.clip(tinv(p.float(),exp['target_mode'],ym,ys).squeeze().cpu().numpy(),0,None)

if RUN_FULL_PIPELINE:
    final_epochs=max(1,round(float(np.median(final_cv.loc[final_cv.experiment==best_name,'best_epoch'])))); FINAL_SEED=2026; print('Final epochs:',final_epochs)
    final_model,ym,ys,xs,elapsed=train_full(best_exp,FINAL_SEED,final_epochs,FINAL_RETRAIN_STRIDE); pred=predict_test(final_model,best_exp,ym,ys,xs); assert pred.shape==(168,) and np.isfinite(pred).all()
    submission=test_df[['time_idx']].copy(); submission['value']=pred; display(submission.head()); display(submission.tail()); submission.to_csv(OUTPUT_DIR/'submission.csv',index=False); np.save(OUTPUT_DIR/'final_predictions.npy',pred)
    torch.save({'model_state_dict':{k:v.detach().cpu() for k,v in final_model.state_dict().items()},'config':asdict(model_cfg(best_exp)),'experiment':best_exp,'target_mean':ym,'target_std':ys,'exog_scaler':xs.state_dict(),'selected_variant':best_name,'seed':FINAL_SEED,'epochs':final_epochs,'parameters':count_params(final_model)},OUTPUT_DIR/'best_autoformer.pt')
    with open(OUTPUT_DIR/'run_manifest.json','w') as f: json.dump({'selected_variant':best_name,'selected_experiment':best_exp,'final_seed':FINAL_SEED,'final_epochs':final_epochs,'parameters':count_params(final_model),'training_minutes':elapsed/60},f,indent=2)
    plt.figure(figsize=(14,4)); plt.plot(train_df.time_idx.iloc[-336:],Y_ALL[-336:],label='observed'); plt.plot(submission.time_idx,submission.value,label='forecast'); plt.legend(); plt.tight_layout(); plt.show(); print('Saved outputs to',OUTPUT_DIR)

## 9. Outputs and selection rule
Primary outputs are `screening_results.csv`, `final_cv_results.csv`, `final_cv_summary.csv`, `final_horizon_rmse.csv`, `submission.csv`, `best_autoformer.pt`, and `run_manifest.json`.

Mean validation RMSE is primary. If candidates are within 0.5% RMSE, the notebook prefers fewer parameters and then fewer epochs. If your assignment PDF gives exact penalty coefficients, replace that tie-break with the exact score formula.